<a href="https://colab.research.google.com/github/NBorthLab/ESACT2026_Transcriptome_Workshop/blob/main/ESACT_2026_Transcriptome_Workshop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ESACT 2026 Transcriptome Workshop

## Setup

In [ ]:
# Install dependencies on the cloud server
!pip install pydeseq2 matplotlib seaborn --quiet

# Import packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats
from sklearn.decomposition import PCA

sns.set(style="whitegrid")


## Data loading

We are now downloading the raw data as well as the metadata.

Next, we want to load the data we just downloaded into Python.

In [ ]:
metadata = pd.read_csv(
  "https://raw.githubusercontent.com/NBorthLab/ESACT2026_Transcriptome_Workshop/refs/heads/main/metadata.csv"
)

metadata

In [ ]:
raw_counts = pd.read_table(
  "https://raw.githubusercontent.com/NBorthLab/ESACT2026_Transcriptome_Workshop/refs/heads/main/counts.tsv",
  index_col = 0
)

raw_counts

## Preprocessing data

In [ ]:
# Set sample IDs as index
metadata.set_index("SampleAccession", inplace=True)
metadata

In [ ]:
# Make sure counts are of type 'integer'
raw_counts = raw_counts.astype(int)

### Gene filtering

Gene with very low expression are inherently noisy in RNA-seq. Additionally, they don't hold statistically valuable information. In order to reduce dimensionality, and therefore strengthen our analysis, we will filter lowly expressed genes out.

In [ ]:
# Number of biological replicates
n_bio_reps = 3

# Is the gene count in a sample more than 10?
counts_greater_10 = raw_counts > 10

# Genes expressed with counts > 10 in more than 3 samples
is_expressed = counts_greater_10.sum(axis = 1) > 3

In [ ]:
counts = raw_counts.loc[is_expressed, :]
counts

## Build and fit the DESeq2 model

In [ ]:
# Initialize DESeq2 inference
inference = DefaultInference(n_cpus=8)

#  Build DESeq2 dataset
dds = DeseqDataSet(
    counts=counts.T,
    metadata=metadata,
    design_factors="Condition",
    refit_cooks=True,
    inference=inference
)

# Run normalization and build DESeq2 model
dds.deseq2()

## Data exploration

In [ ]:
# Perform variance-stabilizing transformation
dds.vst()

In [ ]:
transformed_counts = dds.layers['vst_counts']

### PCA

In [ ]:
X = np.asarray(transformed_counts)

pca = PCA(n_components = 2)
pca_result = pca.fit_transform(X)
explained_variance = pca.explained_variance_ratio_.round(2) * 100

In [ ]:
#plt.scatter(pca_result[:, 0], pca_result[:, 1], c = ["red"] * 3 + ["black"] * 3)
plt.scatter(pca_result[0:3, 0], pca_result[0:3, 1], c = "black")
plt.scatter(pca_result[3:6, 0], pca_result[3:6, 1], c = "red")
plt.annotate("Host", (pca_result[2, 0], pca_result[2, 1]), textcoords="offset points", xytext=(0,10), ha='center')
plt.annotate("Producer", (pca_result[4, 0], pca_result[4, 1]), textcoords="offset points", xytext=(0,10), ha='center')
plt.xlabel("PC1 ({}% explained variance)".format(explained_variance[0]))
plt.ylabel("PC2 ({}% explained variance)".format(explained_variance[1]))
plt.xlim(-36, 36)
plt.ylim(-36, 36)
plt.show()

## Differential gene expression analysis

In [ ]:
# Compute stats
statistics = DeseqStats(dds, contrast = ["Condition", "producer", "host"])
statistics.summary()

results = statistics.results_df

Significantly differentially expressed genes are defined as
$$
p_{adj} < 0.05
$$
and
$$
|\log_2FC| > 1.
$$


### Bonus: _Your turn!_

**What happens when you change these values?** 

Experiment and try it out: change the `significance_level` and `fc_cutoff` and run subsequent code cells again to change the results.

How many genes are differentially expressed under more strict parameters such as $\alpha = 0.01$ and $|\log_2FC| > 2$.

In [ ]:
significance_level = 0.05
fc_cutoff = 1

In [ ]:
def define_significance(gene):
    if gene["padj"] < significance_level and gene["log2FoldChange"] > fc_cutoff:
        return "up"
    elif gene["padj"] < significance_level and gene["log2FoldChange"] < -fc_cutoff:
        return "down"
    else:
        return "ns"

results["regulation"] = results.apply(define_significance, axis = 1)
results["regulation"].value_counts()

In [ ]:
results[results["regulation"] == "up"].to_csv("producer_vs_host_upregulated_genes.csv")
results[results["regulation"] == "down"].to_csv("producer_vs_host_downregulated_genes.csv")

In [ ]:
#volcano plot
plt.figure(figsize=(7,5))
sns.scatterplot(
    x = 'log2FoldChange',
    y = -np.log10(results['padj']),
    data = results, hue='regulation',
    palette = {'up': 'red','down': 'blue', 'ns': 'grey'},
    alpha=0.5
)
plt.title('Producer vs Host', fontsize=15, weight='bold')
plt.xlabel('Log2 Fold Change', fontsize=13, weight='bold')
plt.ylabel('-Log10 p-value', fontsize=13, weight='bold')
plt.axhline(y = -np.log10(significance_level), color='grey', linestyle='--')
plt.axvline(x = fc_cutoff, color='grey', linestyle='--')
plt.axvline(x = -fc_cutoff, color='grey', linestyle='--')
plt.legend(title = 'Regulation')
plt.tight_layout()
plt.show()

## **Functional Enrichment Analysis**

In [ ]:
!pip install gseapy

In [ ]:
import gseapy

### Conversion to mouse

In [ ]:
orthologs = pd.read_csv("https://raw.githubusercontent.com/NBorthLab/ESACT2026_Transcriptome_Workshop/refs/heads/main/orthos.csv")
orthologs = orthologs.set_index("hamster")

In [ ]:
results_ortho = results.join(orthologs)

### Defining gene sets

**Background genes** are important for statistical testing in overrepresentation analysis! These constitute all genes that were subjected to differential gene expression analysis, i.e. the set of genes after gene filtering, before DGE.

In [ ]:
background_genes = results_ortho.mouse.dropna().to_list()

upregulated_genes = results_ortho.query("regulation == 'up'")

downregulated_genes = results_ortho.query("regulation == 'down'")

Here, we download the functional gene sets we want to compare against.

**Gene ontology** terms that describe biological processes that genes (or rather their corresponding proteins) participate in.

**KEGG pathways** that genes are involved in.

In [ ]:
!wget https://raw.githubusercontent.com/NBorthLab/ESACT2026_Transcriptome_Workshop/refs/heads/main/GO_Biological_Process_2026.gmt
!wget https://raw.githubusercontent.com/NBorthLab/ESACT2026_Transcriptome_Workshop/refs/heads/main/KEGG_2019_Mouse.gmt

### Overrepresentation analysis

In [ ]:
go_enrich_downreg = gseapy.enrichr(
    gene_list = downregulated_genes.mouse.dropna(),
    gene_sets = ["./GO_Biological_Process_2026.gmt", "./KEGG_2019_Mouse.gmt"],
    background = background_genes,
    outdir = None,
    cutoff = 0.05,
    verbose = True
)

go_enrich_downreg.results.sort_values(by = "Adjusted P-value").head()

In [ ]:
gseapy.dotplot(go_enrich_downreg.results)

### Bonus: _Your turn!_

Repeat the functional enrichment analysis as above but with the upregulated genes now!

- Change the `gene_list` parameter to the set of upregulated genes (Don't forget to use the `mouse` genes and remove non-converted genes with `dropna()`; See above)
- Save the enrichment in a new variable `go_enrich_upreg`
- Show the `results`, sorted by "Adjusted P-value" and show the first few rows with `.head()` (See above)
- Finally, create a `dotplot` for the enriched terms in upregulated genes.

**How many enriched terms were found in gene ontology and KEGG pathways?**

In [ ]:
# Insert your code here!